In [7]:
from sklearn.datasets import load_iris
data = load_iris()
print(data['data'].shape)
print('-----')
print(data['target'].shape)

(150, 4)
-----
(150,)


In [8]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

data = load_iris()
X, y = data['data'], data['target']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3)

print(X_train.shape,y_train.shape)
print('-----')
print(X_test.shape, y_test.shape)

(105, 4) (105,)
-----
(45, 4) (45,)


In [9]:
from sklearn.ensemble import  RandomForestClassifier

data = load_iris()
X, y = data['data'], data['target']
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.3)

model = RandomForestClassifier()
model.fit(X_train, y_train)
score = model.score(X_test, y_test)
print(score)

0.9777777777777777


In [10]:
from sklearn.metrics import f1_score

y_predict = model.predict(X_test)
print(y_predict)

[1 2 2 1 2 2 1 2 0 2 0 1 0 0 0 1 2 0 1 0 0 1 0 0 1 2 2 0 0 0 0 1 0 2 0 2 1
 2 0 1 1 2 1 0 0]


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms

In [31]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 20, 5, 1)
        self.pool1 = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(20,50,5,1)
        self.pool2 = nn.MaxPool2d(2,2)
        self.fc1 = nn.Linear(4 * 4 * 50, 500)
        self.fc2 = nn.Linear(500, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = x.view(-1, 4 * 4 * 50)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return F.log_softmax(x, dim=1)

In [32]:
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('./resource', train=True,download=True,
                   transform=transforms.ToTensor()),
    batch_size=5000,shuffle=True)

In [33]:
test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('./resource', train=False,
                   transform=transforms.ToTensor()),
    batch_size=5000,shuffle=True)

In [34]:
model = Model()
optimizer = optim.SGD(model.parameters(),lr=0.01,momentum=0.5)

In [36]:
model.train()
for epoch in range(1, 21):
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 2 == 0:
            print('Train Epoch: {} [{} / {} ({:.0f}%)]\tLoss:{:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx/ len(train_loader), loss.item()))

Train Epoch: 1 [0 / 60000 (0%)]	Loss:2.303224
Train Epoch: 1 [10000 / 60000 (17%)]	Loss:2.303800
Train Epoch: 1 [20000 / 60000 (33%)]	Loss:2.302624
Train Epoch: 1 [30000 / 60000 (50%)]	Loss:2.301275
Train Epoch: 1 [40000 / 60000 (67%)]	Loss:2.301037
Train Epoch: 1 [50000 / 60000 (83%)]	Loss:2.298085
Train Epoch: 2 [0 / 60000 (0%)]	Loss:2.298375
Train Epoch: 2 [10000 / 60000 (17%)]	Loss:2.297858
Train Epoch: 2 [20000 / 60000 (33%)]	Loss:2.296685
Train Epoch: 2 [30000 / 60000 (50%)]	Loss:2.295715
Train Epoch: 2 [40000 / 60000 (67%)]	Loss:2.294013
Train Epoch: 2 [50000 / 60000 (83%)]	Loss:2.292457
Train Epoch: 3 [0 / 60000 (0%)]	Loss:2.292614
Train Epoch: 3 [10000 / 60000 (17%)]	Loss:2.290227
Train Epoch: 3 [20000 / 60000 (33%)]	Loss:2.289664
Train Epoch: 3 [30000 / 60000 (50%)]	Loss:2.288683
Train Epoch: 3 [40000 / 60000 (67%)]	Loss:2.287563
Train Epoch: 3 [50000 / 60000 (83%)]	Loss:2.287941
Train Epoch: 4 [0 / 60000 (0%)]	Loss:2.285199
Train Epoch: 4 [10000 / 60000 (17%)]	Loss:2.285268


In [37]:
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in test_loader:
        output= model(data)
        test_loss = test_loss + F.nll_loss(output, target, reduction='sum').item()
        pred = output.argmax(dim=1, keepdims=True) #pred等同於prediction
        correct = correct + pred.eq(target.view_as(pred)).sum().item() #correct應該系糾正之意，sum是總和

test_loss = test_loss / len(test_loader.dataset)

print('\nTest set:Average loss: {:.4f}, Accuracy:{}/{} ({:.0f}%)\n'.format(
    test_loss,correct,len(test_loader.dataset),
    100. * correct / len(test_loader.dataset)))


Test set:Average loss: 0.6397, Accuracy:8520/10000 (85%)



In [38]:
torch.save(model, "sklearn meteorology.ipynb")